# Faruq-v3 BHCL — hierarchical contrastive screening

Seed-42 breadth discovery. BHCL uses the formal SNI-21 `entity_family → 21 leaf` hierarchy on native YOLO26 one-to-many TAL positives. DETR-specific query decoupling is not transferred. The auxiliary projection/prototypes run only during training; inference remains native YOLO26. Locked test is never restored/opened.

In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)
import os,shutil,subprocess,sys,tarfile,time
from pathlib import Path
REPO=Path('/content/coffee-bean-detection'); BRANCH='agent/bhcl-hierarchical-contrastive-screening'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
cmd=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(3):
    r=subprocess.run(cmd)
    if r.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==2: raise RuntimeError('clone gagal')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); os.chdir(REPO)


In [ ]:
import torch
from coffee_detector.drive_project import require_project_artifact,resolve_drive_project_root
assert torch.cuda.is_available(),'Aktifkan GPU'
PROJECT_ROOT=resolve_drive_project_root(required_relative_paths=('bundles/faruq-development-v3-grouped.tar','experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt','experiments/faruq-v3-acmc-optimization-control-v1/val_reports/D0FT_seed42_val.json','experiments/faruq-v3-acmc-optimization-control-v1/val_reports/acmc1_optimization_control_seed42.json'))
ARCHIVE=require_project_artifact(PROJECT_ROOT,'bundles/faruq-development-v3-grouped.tar')
D0=require_project_artifact(PROJECT_ROOT,'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt')
D0FT=require_project_artifact(PROJECT_ROOT,'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/D0FT_seed42_val.json')
ACMC1=require_project_artifact(PROJECT_ROOT,'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/acmc1_optimization_control_seed42.json')
DATA_ROOT=Path('/content/faruq-development-v3-grouped')
if not (DATA_ROOT/'faruq_grouped_summary.json').is_file():
    with tarfile.open(ARCHIVE,'r') as a: a.extractall('/content',filter='data')
GROUPED=DATA_ROOT/'faruq_grouped_summary.json'; assert GROUPED.is_file(); assert not (DATA_ROOT/'test').exists()
OUTPUT=PROJECT_ROOT/'experiments/faruq-v3-bhcl-entity-family-screening-v1'
print(torch.cuda.get_device_name(0),OUTPUT)


In [ ]:
cmd=[sys.executable,'-m','pytest','-q','tests/test_bhcl.py']
print('STATIC CHECK:',' '.join(cmd)); subprocess.run(cmd,cwd=REPO,check=True)


In [ ]:
cmd=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_bhcl_screening','--data-root',str(DATA_ROOT),'--grouped-summary',str(GROUPED),'--d0-checkpoint',str(D0),'--d0ft-report',str(D0FT),'--acmc1-report',str(ACMC1),'--output-root',str(OUTPUT),'--seed','42','--device','0','--authorize-training']
print('MENJALANKAN:',' '.join(cmd),flush=True)
p=subprocess.run(cmd,cwd=REPO,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT); print(p.stdout)
if p.returncode: raise RuntimeError(f'BH1 gagal {p.returncode}')


In [ ]:
import json,pandas as pd
from IPython.display import display
SUMMARY=OUTPUT/'val_reports/bh1_seed42_screening.json'; r=json.loads(SUMMARY.read_text())
assert r['test_opened'] is False and r['test_images_accessed'] is False
rows=[{'model':name,**m} for name,m in r['results'].items()]
display(pd.DataFrame(rows).style.format({'macro_map50_95':'{:.2%}','bottom3_class_map50_95':'{:.2%}','worst_class_map50_95':'{:.2%}'}))
print('BH1 vs D0FT:',r['deltas']['BH1_vs_D0FT']); print('BH1 vs ACMC1:',r['deltas']['BH1_vs_ACMC1'])
print('CRITERIA:',r['criteria']); print('DECISION:',r['decision']); print('SUMMARY:',SUMMARY)
